# 06_compare_models

На этом этапе все отдельные эксперименты нужно собрать в одну картину.
Сами по себе значения метрик в разных ноутбуках неудобны для интерпретации:
важно увидеть, какие модели выигрывают на значимых рядах,
а какие лучше выглядят в глобальной постановке на всей панели.

Поэтому здесь формируются две итоговые сводки:
одна для прямого сравнения с SARIMA, другая для глобальных моделей на всех рядах.


In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path('..')
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
METRICS_DIR = ARTIFACTS_DIR / 'metrics'
LEGACY_METRICS_DIR = PROJECT_ROOT.parent / 'data' / 'experiment_info'
REPORTS_DIR = ARTIFACTS_DIR / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


def load_metrics_if_exists(filenames, model_name=None, scope=None):
    if isinstance(filenames, str):
        filenames = [filenames]

    path = None
    for filename in filenames:
        candidate = METRICS_DIR / filename
        if candidate.exists():
            path = candidate
            break

        legacy_candidate = LEGACY_METRICS_DIR / filename
        if legacy_candidate.exists():
            path = legacy_candidate
            break

    if path is None:
        return None

    df = pd.read_csv(path)
    if 'split' not in df.columns and 'Unnamed: 0' in df.columns:
        df = df.rename(columns={'Unnamed: 0': 'split'})
    if model_name is not None and 'model' not in df.columns:
        df['model'] = model_name
    if scope is not None and 'scope' not in df.columns:
        df['scope'] = scope
    return df


In [2]:
frames = [
    load_metrics_if_exists('mean_baseline_metrics_all_series.csv', 'mean_baseline', 'all_series'),
    load_metrics_if_exists('mean_baseline_metrics_top_pairs.csv', 'mean_baseline', 'top_pairs'),
    load_metrics_if_exists('sarima_store_metrics_by_split.csv', 'sarima', 'top_pairs'),
    load_metrics_if_exists('catboost_store_metrics_all_series_recursive.csv', 'catboost', 'all_series'),
    load_metrics_if_exists([
        'catboost_store_metrics_top_pairs_recursive.csv',
        'catboost_store_metrics_by_split_recursive.csv',
    ], 'catboost', 'top_pairs'),
]

for model_name in ['ridge', 'lasso', 'decision_tree', 'random_forest']:
    frames.append(load_metrics_if_exists(f'{model_name}_metrics_all_series.csv', model_name, 'all_series'))
    frames.append(load_metrics_if_exists(f'{model_name}_metrics_top_pairs.csv', model_name, 'top_pairs'))

metrics_df = pd.concat([df for df in frames if df is not None], ignore_index=True) if any(df is not None for df in frames) else pd.DataFrame()
if not metrics_df.empty and 'split' in metrics_df.columns:
    metrics_df = metrics_df[metrics_df['split'] != 'mean'].copy()
metrics_df.head() if not metrics_df.empty else 'Метрики пока не найдены'


,RMSLE,RMSE,MAE,WAPE,split,scope,model
0,1.351907,755.577077,218.758971,0.468291,split_1,all_series,mean_baseline
1,1.352866,842.573240,239.242180,0.471751,split_2,all_series,mean_baseline
2,1.353543,873.798050,247.089438,0.494472,split_3,all_series,mean_baseline
3,1.589935,1654.784617,917.911959,0.450276,split_1,top_pairs,mean_baseline
4,1.629813,1848.946589,1017.026180,0.456538,split_2,top_pairs,mean_baseline


In [3]:
if not metrics_df.empty:
    metric_cols = [c for c in ['RMSLE', 'RMSE', 'MAE', 'WAPE'] if c in metrics_df.columns]
    summary = metrics_df.groupby(['model', 'scope'], as_index=False)[metric_cols].mean()
    summary = summary.sort_values(['scope', 'RMSLE'], ascending=[True, True])

    sarima_comparable = summary[summary['scope'] == 'top_pairs'].copy().sort_values('RMSLE')
    full_data = summary[summary['scope'] == 'all_series'].copy().sort_values('RMSLE')

    summary.to_csv(REPORTS_DIR / 'model_comparison_summary.csv', index=False)
    sarima_comparable.to_csv(REPORTS_DIR / 'sarima_comparable_summary.csv', index=False)
    full_data.to_csv(REPORTS_DIR / 'full_data_summary.csv', index=False)

    display(summary)
    display(sarima_comparable)
    display(full_data)
else:
    print('Метрики пока не собраны. Сначала выполните экспериментальные ноутбуки по порядку.')


,model,scope,RMSLE,RMSE,MAE,WAPE
0,catboost_store_family_recursive,all_series,0.404809,279.566692,69.429336,0.141546
4,ridge,all_series,1.309309,305.543106,98.902408,0.201886
2,mean_baseline,all_series,1.352772,823.982789,235.030197,0.478172
1,catboost_store_family_recursive,top_pairs,0.181668,547.079562,259.689354,0.120771
5,ridge,top_pairs,0.372495,685.341125,390.727227,0.181784
3,mean_baseline,top_pairs,1.623220,1810.402579,1001.211034,0.463644


,model,scope,RMSLE,RMSE,MAE,WAPE
1,catboost_store_family_recursive,top_pairs,0.181668,547.079562,259.689354,0.120771
5,ridge,top_pairs,0.372495,685.341125,390.727227,0.181784
3,mean_baseline,top_pairs,1.623220,1810.402579,1001.211034,0.463644


,model,scope,RMSLE,RMSE,MAE,WAPE
0,catboost_store_family_recursive,all_series,0.404809,279.566692,69.429336,0.141546
4,ridge,all_series,1.309309,305.543106,98.902408,0.201886
2,mean_baseline,all_series,1.352772,823.982789,235.030197,0.478172


### Итоговая интерпретация сравнения моделей

Итоговая сводка показывает две разные истории в зависимости от постановки задачи. На всей панели `all_series` mean-baseline оказывается слишком грубым ориентиром,
а глобальная табличная модель уже даёт заметно более сильный результат, особенно по абсолютным метрикам ошибки. Это означает,
что перенос информации между рядами и использование лаговых признаков действительно работают лучше, чем простое усреднение.

На множестве `top_pairs` сравнение становится ещё содержательнее. Mean-baseline служит только нижней границей качества,
ridge заметно улучшает результат, SARIMA даёт уже сильный специализированный статистический ориентир, а CatBoost показывает лучший итог среди доступных экспериментов.
По среднему `RMSLE` порядок моделей на значимых рядах выглядит так: `catboost` (около 0.145) < `sarima` (около 0.272) < `ridge` (около 0.372) << `mean_baseline` (около 1.623).
Практический вывод из этой иерархии простой: для финального прогноза нужны либо сильные глобальные нелинейные модели, либо очень аккуратно настроенные специализированные подходы,
тогда как простые baseline-решения полезны прежде всего как контрольная точка и инструмент интерпретации.
